In [1]:
# ==========================================
# 1. PREPARACIÓ DE L'ENTORN MODERN
# ==========================================
using Pkg
Pkg.activate("/home/antonibancells/Desktop/Codi/TFM/ProvesModernes")

using ITensors
using ITensorMPS
using Plots

  Activating project at `~/Desktop/Codi/TFM/ProvesModernes`
Precompiling packages...
   1084.3 ms  ✓ libdrm_jll
    427.5 ms  ✓ Dbus_jll
    451.7 ms  ✓ xkbcommon_jll
    509.1 ms  ✓ Wayland_jll
    929.8 ms  ✓ HarfBuzz_jll
    926.2 ms  ✓ libva_jll
    879.7 ms  ✓ Vulkan_Loader_jll
    883.3 ms  ✓ libass_jll
    888.0 ms  ✓ Pango_jll
   1000.0 ms  ✓ libdecor_jll
   1709.0 ms  ✓ GLFW_jll
   4229.1 ms  ✓ FFMPEG_jll
    317.6 ms  ✓ FFMPEG
   5702.3 ms  ✓ Qt6Base_jll
    425.7 ms  ✓ Qt6ShaderTools_jll
    443.9 ms  ✓ Qt6Svg_jll
   1754.7 ms  ✓ Qt6Declarative_jll
   2666.8 ms  ✓ GR_jll
    429.2 ms  ✓ Qt6Wayland_jll
   2428.3 ms  ✓ GR
  42914.1 ms  ✓ Plots
  21 dependencies successfully precompiled in 58 seconds. 128 already precompiled.
[ Info: Precompiling Plots [91a5bcdd-55d7-5caf-9e0b-520d859cae80]
Precompiling packages...
    291.8 ms  ✓ InlineStrings → ParsersExt
  1 dependency successfully precompiled in 0 seconds. 4 already precompiled.
[ Info: Precompiling IJuliaExt [64482eec-cc57

In [2]:
using ITensorTDVP

function solve_tdvp(N, J, Δ, h_profile, t_total, δt)
    sites = siteinds("S=1/2", N; conserve_qns=true)

    # --- EL MISMO HAMILTONIANO DE DMRG ---
    os = OpSum()
    for j in 1:N
        os += h_profile[j], "Sz", j
    end
    for j in 1:(N - 1)
        if j == 1 # Condición de Robin
            os += 0.5 * 0.5 * J, "S+", j, "S-", j+1
            os += 0.5 * 0.5 * J, "S-", j, "S+", j+1
            os += 0.5 * J * 0.8 * Δ, "Sz", j, "Sz", j+1
            os += 0.2, "Sz", 1 
        elseif j == N - 1 # Condición de Dirichlet
            os += 0.5 * J, "S+", j, "S-", j+1
            os += 0.5 * J, "S-", j, "S+", j+1
            os += J * Δ, "Sz", j, "Sz", j+1
            os += 5.0, "Sz", N 
        else # Bulk
            os += 0.5 * J, "S+", j, "S-", j+1
            os += 0.5 * J, "S-", j, "S+", j+1
            os += J * Δ, "Sz", j, "Sz", j+1
        end
    end
    H = MPO(os, sites)

    # --- ESTADO INICIAL (Pulso central) ---
    state = fill("Dn", N)
    state[N ÷ 2] = "Up"
    psi = MPS(sites, state)

    # --- EVOLUCIÓN CON TDVP ---
    # t_total: tiempo final, δt: paso de tiempo
    # time_step: el paso que TDVP toma internamente
    times = 0.0:δt:t_total
    data = zeros(length(times), N)

    println("Iniciando TDVP...")
    for (idx, t) in enumerate(times)
        data[idx, :] = expect(psi, "Sz")
        
        # tdvp evoluciona el estado psi por un intervalo δt
        # 'nsweeps=1' suele bastar para pasos de tiempo pequeños
        psi = tdvp(H, -im * δt, psi; 
                   nsweeps=1, 
                   cutoff=1E-10, 
                   maxdim=100)
    end

    return times, data
end

solve_tdvp (generic function with 1 method)